### 1. 빵셀러 데이터셋 불러오기

In [8]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터

import numpy as np
import pandas as pd

np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


### 2. 발견한 품질 문제

In [19]:
# 2-1. 품질 리포트 (결측·중복·이상치 한눈에)
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

   
    # 글자 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    import warnings
    suspicious_datetime = []
    for col in df.select_dtypes(exclude="number").columns:   # 숫자가 아닌 컬럼
        try:
            with warnings.catch_warnings():                  # 경고 메시지 숨기기
                warnings.simplefilter("ignore")
                parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass

    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
store_id,str,0,0.00,7,NaN,False
date_str,str,0,0.00,9,NaN,False
item,str,0,0.00,6,NaN,False
quantity,int64,0,0.00,5,14.56,False
unit_price,int64,0,0.00,5,0.00,False
revenue,float64,60,2.77,26,17.30,False


In [11]:
# 2-2. 표기 혼재 확인
print("[store_id] 공백 포함 값 (repr로 공백이 보이게)")
for s in bakery['store_id'].unique():
    print(repr(s))

print("\n[item] '케익' / '케이크' 혼재")
print(bakery['item'].value_counts())

print("\n[date_str] 같은 날짜가 3가지 형식으로 기록됨")
print(bakery['date_str'].value_counts())

[store_id] 공백 포함 값 (repr로 공백이 보이게)
'S01'
'S02'
'S03'
'S04'
'S05'
'S06'
' S01 '

[item] '케익' / '케이크' 혼재
item
쿠키      446
샌드위치    443
식빵      435
케이크     417
크루아상    415
케익        8
Name: count, dtype: int64

[date_str] 같은 날짜가 3가지 형식으로 기록됨
date_str
20250401      261
2025/05/15    256
20250620      255
2025-04-01    242
2025-06-20    238
2025-05-15    233
20250515      231
2025/06/20    228
2025/04/01    220
Name: count, dtype: int64


In [12]:
# 2-3. revenue 검증: quantity × unit_price와 맞는지, 음수가 있는지
calc = bakery['quantity'] * bakery['unit_price']
mismatch = bakery[bakery['revenue'].notna() & (bakery['revenue'] != calc)]
print("계산값과 안 맞는 행 수:", len(mismatch))
print(mismatch)

계산값과 안 맞는 행 수: 1
  store_id    date_str  item  quantity  unit_price  revenue
5      S06  2025/05/15  샌드위치         5        4500 -45000.0


### 3. 처리 결정과 근거

In [13]:
# 3. 정제 (단계마다 행 수 출력)
clean = bakery.copy()
print("원본:", len(clean))

# 1) 중복 제거
clean = clean.drop_duplicates()
print("1) 중복 제거 후:", len(clean))

# 2) store_id 공백 제거
clean['store_id'] = clean['store_id'].str.strip()

# 3) '케익' → '케이크' 통일
clean['item'] = clean['item'].replace({'케익': '케이크'})

# (KPI용) 날짜 형식 통일 + 월 컬럼 만들기
clean['date'] = pd.to_datetime(
    clean['date_str'].str.replace(r'[-/]', '', regex=True), format='%Y%m%d')
clean['month'] = clean['date'].dt.to_period('M').astype(str)

# 4) revenue 음수 → 별도 보관
refunds_bakery = clean[clean['revenue'] < 0]
clean = clean[~(clean['revenue'] < 0)]
print("4) 음수 분리:", len(refunds_bakery), "건 →", len(clean))

# 5) revenue 결측 제거
print("5) 남은 결측:", clean['revenue'].isna().sum(), "건")
clean = clean.dropna(subset=['revenue'])
print("최종:", len(clean))

bakery_clean = clean

원본: 2164
1) 중복 제거 후: 1843
4) 음수 분리: 1 건 → 1842
5) 남은 결측: 58 건
최종: 1784


### 4. 주요 KPI 결과

In [14]:
# 4-1. 월·매장별 매출
store_month = bakery_clean.pivot_table(
    index='store_id', columns='month', values='revenue', aggfunc='sum')
print(store_month)
print("\n가장 큰 값:", store_month.stack().idxmax(), store_month.max().max())

month       2025-04    2025-05    2025-06
store_id                                 
S01       6834000.0  7243500.0  5884000.0
S02       7531500.0  6768500.0  7996500.0
S03       7424500.0  4548000.0  7297500.0
S04       6471500.0  7105000.0  6744500.0
S05       8681500.0  4102000.0  4960000.0
S06       9156000.0  5095500.0  6796500.0

가장 큰 값: ('S06', '2025-04') 9156000.0


In [15]:
# 4-2. 상품별 KPI
item_kpi = (bakery_clean.groupby('item')
            .agg(avg_price=('unit_price', 'mean'),
                 total_revenue=('revenue', 'sum'),
                 n_orders=('revenue', 'size'))
            .round(0)
            .sort_values('total_revenue', ascending=False))
item_kpi

,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335


### 5. 한계와 후속 작업

In [16]:
# 5-1. quantity=50 주문이 매출에서 차지하는 비중
big = bakery_clean['quantity'] == 50
print("50개 주문 매출 비중:",
      round(bakery_clean.loc[big, 'revenue'].sum() / bakery_clean['revenue'].sum() * 100, 1), "%")

# 50 제외하면 상품 순위가 어떻게 되는지
print("\n[50 제외] 상품별 매출:")
print(bakery_clean[~big].groupby('item')['revenue'].sum().sort_values(ascending=False))

50개 주문 매출 비중: 80.4 %

[50 제외] 상품별 매출:
item
쿠키      4940000.0
샌드위치    4797000.0
크루아상    4692500.0
케이크     4688500.0
식빵      4572500.0
Name: revenue, dtype: float64


In [17]:
# 5-2. 실제 기록된 날짜는 3일뿐
print(bakery_clean['date'].dt.date.unique())

[datetime.date(2025, 4, 1) datetime.date(2025, 6, 20)
 datetime.date(2025, 5, 15)]


### 🍞 빵셀러 베이커리 체인 운영 로그 최종 보고서 🥐

### 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그
- 기간: 2025-04-01 ~ 2025-06-30 (3개월)
- 원본 행 수 / 정제 후 행 수: 2164행 → 1784행

### 2. 발견한 품질 문제
- 결측: revenue, 60건, 2.77%
- 중복: 321건
- 이상치: revenue 17.30%, quantity 14.56% (IQR 기준)
- 표기 혼재:  - store_id: 'S01'과 ' S01 '(앞뒤 공백)이 따로 집계됨
  - item: '케익'(8건)과 '케이크'가 섞여 있음
  - date_str: 같은 날짜가 3가지 형식으로 적혀 있음 (20250401 / 2025-04-01 / 2025/04/01)

### 3. 처리 결정과 근거
1. 중복 행 (321)건 → 제거 (같은 거래가 두 번 기록돼 매출이 부풀려짐)
2. store_id 공백 → 공백 제거 (같은 매장이 다른 매장으로 집계됨)
3. item '케익' → '케이크' → 통일 (같은 상품이 두 개로 나뉘어 집계됨)
4. revenue 음수 → 별도 보관 1건
5. revenue 결측 (58)건 → 제거 (이유: 비율이 낮아 제거해도 결과에 영향이 작음) *5번이 60이 아니라 58인 이유: 결측 60건 중 2건은 중복 행이라서 1번에서 이미 지워졌어. 그래서 실제로 지운 건 58건.*

### 4. 주요 KPI 결과
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: S06, 2025년 4월 (9,156,000원)
- (상품 관점) 매출 1위 상품: 쿠키  (26,690,000원)

### 5. 한계와 후속 작업
-  quantity가 50인 주문이 315건 있고, 이 주문들이 전체 매출의 80.4%를 차지함.
  진짜 대량 주문인지 오타(5→50)인지 확인 필요
- revenue 음수 1건(-45,000원)은 계산값(22,500원)의 2배라서,
  환불인지 입력 오류인지 원인 확인 필요
- 기록된 날짜가 4/1, 5/15, 6/20 세 날뿐이라 일별 추이는 분석할 수 없음